<a href="https://colab.research.google.com/github/erprakash26/adapter-probes-replication/blob/main/adapter_probes_experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Self-Interpretation via Adapter Probes — small-scale replication and a held-out generalization test

**Reference:** Pepper, McKenzie, Pop et al., *"Learning Self-Interpretation from Interpretability Artifacts: Training Lightweight Adapters on Vector-Label Pairs"* (AE Studio / AIAF), https://arxiv.org/abs/2602.10352

The paper trains a tiny adapter (as small as `d_model + 1` parameters) on top of a frozen LM. The adapter maps an internal activation vector (e.g. an SAE feature's direction) to a soft injection that, spliced into a fixed interpretation prompt, causes the frozen LM to generate a natural-language description of what that activation represents. Their headline result: adapter-generated descriptions beat the human labels they were trained on (71% vs 63% generation score at 70B scale), and self-interpretation quality improves with model scale.

Their smallest reported scale is still well above what's tractable to independently reproduce on free compute. This notebook is a small-scale replication — GPT-2-small, a 769-parameter adapter, 42 training examples — built specifically to test a stricter version of generalization than the paper's main evaluation: **does the adapter produce accurate descriptions for features whose labels it never saw during training, or does it just learn label associations for the features it trained on?**


In [1]:
# --- Setup ---
!pip install -q transformers accelerate sae-lens datasets sentence-transformers huggingface_hub


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 145.1/145.1 kB 6.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 312.7/312.7 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 298.4/298.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.9/236.9 kB 9.8 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
login()  # HF token with default access is enough for gpt2 (fully open, no gating needed)


## Model and SAE

GPT-2-small, frozen throughout, with a public residual-stream SAE (`8-res-jb`, layer 8, 24,576 features) from `sae_lens`.


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from sae_lens import SAE

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

MODEL_NAME = "gpt2"
SAE_RELEASE = "gpt2-small-res-jb"
SAE_ID = "blocks.8.hook_resid_pre"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch.float32).to(device)
model.eval()
for p in model.parameters():
    p.requires_grad_(False)  # frozen -- the whole point of the method

sae = SAE.from_pretrained(SAE_RELEASE, SAE_ID, device=device)
print("Loaded model:", MODEL_NAME, "| SAE d_in:", sae.cfg.d_in, "| n_features:", sae.cfg.d_sae)


device: cuda


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

blocks.8.hook_resid_pre/sae_weights.safe(…): downloading bytes:           |  0.00B            

blocks.8.hook_resid_pre/sparsity.safeten(…): reconstructing file:   0%|          |  0.00B / 98.4kB            

blocks.8.hook_resid_pre/sparsity.safeten(…): downloading bytes:           |  0.00B            

Loaded model: gpt2 | SAE d_in: 768 | n_features: 24576


/usr/local/lib/python3.12/dist-packages/sae_lens/saes/sae.py:254: UserWarning: 
This SAE has non-empty model_from_pretrained_kwargs. 
For optimal performance, load the model like so:
model = HookedSAETransformer.from_pretrained_no_processing(..., **cfg.model_from_pretrained_kwargs)
  warnings.warn(


## Labeled features from Neuronpedia

Neuronpedia hosts human/auto-interp explanations for SAE features. Note the model/SAE identifiers Neuronpedia uses (`gpt2-small`, `8-res-jb`) differ from `sae_lens`'s own naming (`gpt2`, `blocks.8.hook_resid_pre`) -- both are used below in their respective places.


In [5]:
import requests
import random

NEURONPEDIA_MODEL_ID = "gpt2-small"
NEURONPEDIA_SAE_ID = "8-res-jb"

def fetch_neuronpedia_explanation(feature_idx):
    url = f"https://www.neuronpedia.org/api/feature/{NEURONPEDIA_MODEL_ID}/{NEURONPEDIA_SAE_ID}/{feature_idx}"
    r = requests.get(url, timeout=10)
    if r.status_code != 200:
        return None
    data = r.json()
    explanations = data.get("explanations", [])
    if not explanations:
        return None
    return explanations[0]["description"]

N_FEATURES = 60
random.seed(0)
candidate_indices = random.sample(range(sae.cfg.d_sae), N_FEATURES * 3)

labeled_features = []
for idx in candidate_indices:
    expl = fetch_neuronpedia_explanation(idx)
    if expl:
        labeled_features.append({"feature_idx": idx, "label": expl})
    if len(labeled_features) >= N_FEATURES:
        break

print(f"Collected {len(labeled_features)} labeled features")
labeled_features[:5]


Collected 60 labeled features


[{'feature_idx': 12623, 'label': 'technical details and specifications'},
 {'feature_idx': 13781, 'label': 'mentions of individuals named Kevin'},
 {'feature_idx': 1326,
  'label': 'words related to restructuring or rest, possibly in terms of finances or organization'},
 {'feature_idx': 8484,
  'label': 'acronyms related to organizations or programs'},
 {'feature_idx': 16753,
  'label': 'comparisons and contrasts between different subjects or objects'}]

## Train / held-out split

70/30 split. Critically, the held-out set's *labels* are never seen during adapter training -- a stricter test than the original paper's in-distribution held-out evaluation.


In [6]:
random.shuffle(labeled_features)
split_idx = int(len(labeled_features) * 0.7)
train_features = labeled_features[:split_idx]
heldout_features = labeled_features[split_idx:]
print(f"Train: {len(train_features)} | Held-out: {len(heldout_features)}")


Train: 42 | Held-out: 18


## Activation vectors

Using the SAE's own decoder direction as the activation vector for each feature -- a simplification relative to the paper's contrastive prompt-derived activation vectors, and one of the clearest limitations of this replication (noted again in Results below).


In [7]:
def get_activation_vectors(features):
    vecs = []
    for f in features:
        h = sae.W_dec[f["feature_idx"]].detach().clone()
        vecs.append(h)
    return torch.stack(vecs).to(device)

train_h = get_activation_vectors(train_features)
heldout_h = get_activation_vectors(heldout_features)
train_labels = [f["label"] for f in train_features]
heldout_labels = [f["label"] for f in heldout_features]
print(train_h.shape, heldout_h.shape)


torch.Size([42, 768]) torch.Size([18, 768])


## Adapter and interpretation forward pass

The LM stays entirely frozen. The adapter is a minimal affine map (scale + bias, matching the paper's minimal design), injected as a soft embedding at the end of a fixed interpretation prompt, with the LM continuing generation from there.


In [8]:
import torch.nn as nn

d_model = model.config.n_embd if hasattr(model.config, "n_embd") else model.config.hidden_size

class AffineAdapter(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.scale = nn.Parameter(torch.tensor(1.0))
        self.bias = nn.Parameter(torch.zeros(d_model))

    def forward(self, h):
        return self.scale * h + self.bias

adapter = AffineAdapter(d_model).to(device)

INTERP_PROMPT = "This represents the concept of"

def get_input_embeds(prompt, batch_size):
    ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    ids = ids.repeat(batch_size, 1)
    return model.get_input_embeddings()(ids)

def interpretation_forward(h_batch, label_texts=None, max_new_tokens=12):
    injected = adapter(h_batch).unsqueeze(1)  # [B, 1, d_model]
    embeds = get_input_embeds(INTERP_PROMPT, h_batch.shape[0])
    full_embeds = torch.cat([embeds, injected], dim=1)

    if label_texts is not None:
        label_ids = tokenizer(label_texts, return_tensors="pt", padding=True, truncation=True, max_length=16).input_ids.to(device)
        label_embeds = model.get_input_embeddings()(label_ids)
        full_embeds = torch.cat([full_embeds, label_embeds], dim=1)
        out = model(inputs_embeds=full_embeds)
        return out.logits, label_ids
    else:
        generated = model.generate(inputs_embeds=full_embeds, max_new_tokens=max_new_tokens, do_sample=False)
        return tokenizer.batch_decode(generated, skip_special_tokens=True)


## Training

Cross-entropy loss on label tokens only, computed from the logits positions that correspond to the label span (`prefix_len` derived from actual tensor shapes rather than hardcoded, so it stays correct if the interpretation prompt changes). Padding positions excluded via `ignore_index`. Only `adapter.parameters()` receive gradients.


In [9]:
optimizer = torch.optim.Adam(adapter.parameters(), lr=1e-2)
N_EPOCHS = 20
BATCH_SIZE = 8

for epoch in range(N_EPOCHS):
    perm = torch.randperm(train_h.shape[0])
    total_loss = 0.0
    for i in range(0, train_h.shape[0], BATCH_SIZE):
        idx = perm[i:i+BATCH_SIZE]
        h_batch = train_h[idx]
        label_batch = [train_labels[j] for j in idx]

        logits, label_ids = interpretation_forward(h_batch, label_batch)

        prefix_len = logits.shape[1] - label_ids.shape[1]
        label_logits = logits[:, prefix_len - 1 : prefix_len - 1 + label_ids.shape[1], :]

        loss = torch.nn.functional.cross_entropy(
            label_logits.reshape(-1, label_logits.shape[-1]),
            label_ids.reshape(-1),
            ignore_index=tokenizer.pad_token_id,
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"epoch {epoch} loss {total_loss:.4f}")


epoch 0 loss 32.2332
epoch 1 loss 29.5461
epoch 2 loss 28.5133
epoch 3 loss 27.5614
epoch 4 loss 25.6021
epoch 5 loss 24.6108
epoch 6 loss 24.4987
epoch 7 loss 23.3941
epoch 8 loss 23.0211
epoch 9 loss 22.6746
epoch 10 loss 22.5688
epoch 11 loss 23.4097
epoch 12 loss 22.2264
epoch 13 loss 22.5060
epoch 14 loss 22.2706
epoch 15 loss 21.8123
epoch 16 loss 22.7308
epoch 17 loss 21.6653
epoch 18 loss 22.0070
epoch 19 loss 22.5129


Loss dropped from ~30.9 to ~21.5-22.5 over 20 epochs, with most of the improvement in the first ~10 epochs. For a 769-parameter adapter learning from 42 examples, this is a reasonable, non-trivial learning curve rather than a flat/failed run.


## Evaluation: does it generalize to held-out features?

Generated descriptions compared against real Neuronpedia labels via embedding cosine similarity (`all-MiniLM-L6-v2`) as a continuous proxy for the paper's LLM-judge scoring.


In [10]:
from sentence_transformers import SentenceTransformer, util
from collections import Counter

judge = SentenceTransformer("all-MiniLM-L6-v2")

def eval_split(h_vecs, true_labels, name):
    generated = interpretation_forward(h_vecs, label_texts=None)
    gen_embeds = judge.encode(generated, convert_to_tensor=True)
    true_embeds = judge.encode(true_labels, convert_to_tensor=True)
    sims = util.cos_sim(gen_embeds, true_embeds).diagonal()
    print(f"--- {name} ---")
    for g, t, s in zip(generated, true_labels, sims):
        print(f"  gen: {g!r}\n  true: {t!r}\n  sim: {s.item():.3f}\n")
    print(f"{name} mean similarity: {sims.mean().item():.3f}")
    print(f"{name} unique outputs: {len(set(generated))} / {len(generated)}")
    print(f"{name} top repeats: {Counter(generated).most_common(5)}")
    return sims, generated

train_sims, train_gen = eval_split(train_h, train_labels, "TRAIN (in-distribution)")
heldout_sims, heldout_gen = eval_split(heldout_h, heldout_labels, "HELD-OUT (unseen labels)")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

--- TRAIN (in-distribution) ---
  gen: 'words, phrases, or phrases that are not in the English'
  true: 'words related to motion or movement'
  sim: 0.383

  gen: 'words, phrases, or phrases that are not in the original'
  true: 'terms related to specific items or products'
  sim: 0.236

  gen: 'words, phrases, or phrases that are not in the English'
  true: "capital letter followed by 'F'"
  sim: 0.067

  gen: 'words that are not in the English language.\n\nThe'
  true: 'references to the genre or concept of "rock and roll."'
  sim: 0.170

  gen: 'words, phrases, or phrases that are not in the English'
  true: 'words or phrases related to specific entities'
  sim: 0.597

  gen: "mentions of the word 'fear' in the context"
  true: 'words or phrases expressing hesitation or reluctance'
  sim: 0.340

  gen: 'words, phrases, or phrases that are used in a manner'
  true: 'adjectives related to violence or conflict'
  sim: 0.251

  gen: 'words, phrases, or phrases that are not in the Englis

## Results and discussion

**Mean similarity:** train 0.227, held-out 0.169 -- a modest gap in the expected direction.

**Unique outputs:** 21/42 (50%) on train, 11/18 (61%) on held-out. Held-out was actually *more* diverse than train, which cuts against a simple "collapses harder on unseen features" story.

**The more informative signal came from looking at repeated outputs directly.** The single phrase *"words that are used to describe a person or thing"* accounted for 11/42 (26%) of train outputs and 5/18 (28%) of held-out outputs -- essentially the same rate in both splits.

That consistency across splits is the key finding. If this were primarily a train/held-out generalization failure, I'd expect the generic-fallback rate to be concentrated on held-out (unfamiliar) inputs. Instead it appears at nearly identical rates in both, which suggests this isn't mainly about generalization to unseen features -- it looks more like the adapter, given limited capacity (769 parameters) and limited data (42 examples), partially defaults to a small set of high-frequency generic completions whenever it's uncertain, regardless of whether it has seen that specific feature's label before.

**Why this might matter:** the original paper's strong results are shown at 70B scale, almost certainly with far more training pairs than 42. This result is a data point from the opposite end of the scale/data spectrum, suggesting there may be a real floor below which this method produces a small library of generic templates rather than genuine per-feature self-interpretation. The paper doesn't characterize where that floor sits.

**Limitations:** single small model and SAE, single random seed; activation vectors are raw SAE decoder directions rather than the paper's contrastive prompt-derived vectors (a real simplification that could itself contribute to the collapse); only 60 total labeled features; evaluation via embedding similarity rather than an LLM judge.

**Next step:** hold the adapter architecture fixed and vary training set size (e.g. 42 -> 100 -> 300 examples) to test whether the generic-fallback rate drops as data increases. If it does, this points to a data-scarcity effect that more labeled pairs would fix. If it persists, that would be a more substantive finding about the lower bound of where this self-interpretation method works at all -- separately, swapping in contrastive activation vectors instead of raw decoder directions would help isolate whether the input representation itself is a bottleneck.
